# Week 6 — Convergence Check

Verifies that all five agents are learning before committing to the full multi-regime sweep.

**Three pass/fail checks:**
1. Episode reward trending upward by episode 500 for all agents
2. QR-DQN quantile crossing rate < 5%
3. No agent at inventory bounds more than 5% of steps

**Inputs:** `logs/{agent}_handcrafted_asymmetric_low_vol_seed42/train_history.json`  
**Outputs:** `experiments/w06_convergence/`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import torch

from evaluation.metrics import load_train_history
from evaluation.visualize import Visualizer
from training.evaluate import load_agent, evaluate_checkpoint
from envs.lob_env import LOBMarketMakingEnv

LOG_ROOT  = Path('../logs')
CKPT_ROOT = Path('../checkpoints')
EXP_DIR   = Path('../experiments/w06_convergence')
EXP_DIR.mkdir(parents=True, exist_ok=True)

AGENTS  = ['sarsa', 'dqn', 'ppo', 'qrdqn', 'iqn']
REGIME  = 'low_vol'
ENCODER = 'handcrafted'
REWARD  = 'asymmetric'
SEED    = 42
Q_MAX   = 10

def run_tag(agent):
    return f'{agent}_{ENCODER}_{REWARD}_{REGIME}_seed{SEED}'

viz = Visualizer(log_root=LOG_ROOT, ckpt_root=CKPT_ROOT, out_root=EXP_DIR)
print('Setup complete')

## 1 — Load training histories

In [ ]:
histories = {}
for agent in AGENTS:
    run_dir = LOG_ROOT / run_tag(agent)
    if not run_dir.exists():
        print(f'  [missing] {run_tag(agent)}')
        continue
    try:
        histories[agent] = load_train_history(run_dir)
        n  = len(histories[agent])
        sh = histories[agent]['sharpe'].iloc[-50:].mean()
        print(f'  {agent:8s}: {n} eps, final sharpe={sh:+.4f}')
    except Exception as e:
        print(f'  {agent}: ERROR {e}')
print(f'Loaded {len(histories)}/{len(AGENTS)} agents')

## 2 — Convergence curves

In [ ]:
for metric in ['sharpe', 'final_pnl', 'map']:
    viz.plot_convergence(
        agents=AGENTS, metric=metric, smooth=20,
        regime=REGIME, encoder=ENCODER, reward=REWARD, seed=SEED,
        save=True,
    )

## 3 — Check 1: Reward trending upward by episode 500

In [ ]:
print('CHECK 1: Reward trending upward by ep 500')
print('─' * 50)
check1 = {}
for agent, df in histories.items():
    if len(df) < 500:
        print(f'  {agent:8s}: SKIP (only {len(df)} eps)')
        continue
    early = float(df['final_pnl'].iloc[:100].mean())
    ep500 = float(df['final_pnl'].iloc[500:600].mean())
    check1[agent] = ep500 > early
    print(f'  {agent:8s}: early={early:+.2f} ep500={ep500:+.2f} '
          f'→ {"PASS" if check1[agent] else "FAIL"}')
n = sum(check1.values())
print(f'\n{n}/{len(check1)} PASS')

## 4 — Check 2: QR-DQN quantile crossing rate < 5%

In [ ]:
print('CHECK 2: QR-DQN quantile crossing rate < 5%')
print('─' * 45)
ckpt_path = CKPT_ROOT / run_tag('qrdqn') / 'ep01000.pt'
if not ckpt_path.exists():
    print('Checkpoint not found — run training first')
else:
    agent_qr, _ = load_agent(str(ckpt_path), 'qrdqn', 'handcrafted')
    agent_qr.reset_hidden(batch_size=1)
    rates = []
    for _ in range(500):
        obs = torch.randn(1, 18)
        with torch.no_grad():
            h, _ = agent_qr.online_base.forward(obs.unsqueeze(1))
            Z    = agent_qr.online_head(h[:, -1, :])   # (1, n_actions, N)
        rates.append(float((Z[0].diff(dim=-1) < 0).float().mean()))
    rate = np.mean(rates)
    print(f'  Crossing rate: {rate:.1%} → {"PASS" if rate < 0.05 else "FAIL"}')

## 5 — Check 3: Inventory at bounds < 5%

In [ ]:
print('CHECK 3: Inventory at bounds < 5% of steps')
print('─' * 45)
for agent, df in histories.items():
    if 'inventory_at_bounds' in df.columns:
        rate = float(df['inventory_at_bounds'].iloc[-100:].mean())
        print(f'  {agent:8s}: {rate:.1%} → {"PASS" if rate < 0.05 else "FAIL"}')
    else:
        # Proxy via MAP
        map_val = float(df['map'].iloc[-100:].mean())
        ok = map_val < 0.8 * Q_MAX
        print(f'  {agent:8s}: MAP={map_val:.2f} (proxy) → {"PASS" if ok else "FAIL"}')

## 6 — CVaR vs mean divergence

In [ ]:
print('CVaR vs mean divergence (target > 5%)')
print('─' * 40)
from agents.cvar_policy import CVaRPolicy
for agent_name in ['qrdqn', 'iqn']:
    ckpt = CKPT_ROOT / run_tag(agent_name) / 'ep01000.pt'
    if not ckpt.exists():
        print(f'  {agent_name}: checkpoint not found')
        continue
    base, _ = load_agent(str(ckpt), agent_name, 'handcrafted')
    cvar_ag  = CVaRPolicy(agent=base, alpha=0.25)
    n, diff = 200, 0
    for _ in range(n):
        obs = np.random.randn(18).astype(np.float32)
        if cvar_ag.act(obs, greedy=True) != base.act(obs, greedy=True):
            diff += 1
    rate = diff / n
    print(f'  {agent_name:8s}: {rate:.1%} → {"PASS" if rate > 0.05 else "WARN"}')

## 7 — PPO documentation

In [ ]:
if 'ppo' in histories:
    df = histories['ppo']
    sh = float(df['sharpe'].iloc[-100:].mean())
    pnl = float(df['final_pnl'].iloc[-100:].mean())
    print(f'PPO: sharpe={sh:+.4f}  pnl={pnl:+.4f}')
    print('Converged' if sh > 0 else
          'Unstable — confirms Beysolow (2019). Document as result.')
else:
    print('PPO history not found')

## 8 — Figure 2: Quote skew vs GLFT

In [ ]:
env = LOBMarketMakingEnv(
    reward_type='asymmetric', episode_len=390,
    Q_max=Q_MAX, tick_size=0.01, seed=1000, use_abides=False,
)
skew_data = {}
for agent_name in AGENTS:
    ext  = '.npz' if agent_name == 'sarsa' else '.pt'
    cdir = CKPT_ROOT / run_tag(agent_name)
    cks  = sorted([c for c in cdir.glob(f'*{ext}') if '_meta' not in c.name]) if cdir.exists() else []
    if not cks:
        continue
    try:
        ag, et = load_agent(str(cks[-1]), agent_name, ENCODER)
        res = evaluate_checkpoint(ag, env, et, n_episodes=10, seed=1000)
        sd  = res['skew_data']
        if len(sd['inv_levels']) >= 3:
            skew_data[agent_name] = (sd['inv_levels'], sd['mean_offsets'], sd['std_offsets'])
            print(f'  {agent_name}: sharpe={res["metrics"]["sharpe_mean"]:+.4f}')
    except Exception as e:
        print(f'  {agent_name}: {e}')
env.close()

In [ ]:
if skew_data:
    viz.plot_quote_skew(
        agent_data=skew_data,
        title='Figure 2 — Agent Quote Skew vs GLFT Reference\nlow_vol · handcrafted',
        overlay_baselines=True, save=True,
        save_name='figure2_quote_skew.png', save_subdir='w06_convergence',
    )
else:
    print('No skew data — run training first')

## 9 — Summary table

In [ ]:
rows = []
for agent, df in histories.items():
    rows.append({
        'agent':        agent,
        'n_episodes':   len(df),
        'sharpe_final': round(float(df['sharpe'].iloc[-100:].mean()), 4),
        'map_final':    round(float(df['map'].iloc[-100:].mean()), 4),
        'pnl_final':    round(float(df['final_pnl'].iloc[-100:].mean()), 4),
        'check1':       check1.get(agent, 'N/A'),
    })
summary = pd.DataFrame(rows).sort_values('sharpe_final', ascending=False)
display(summary)
summary.to_csv(EXP_DIR / 'convergence_summary.csv', index=False)
print(f'Saved → {EXP_DIR}/convergence_summary.csv')